# RS/OF 2D-Gaussian fit-quality viewer

Interactive QC for the trial-averaged, treadmill (closed-loop) `gaussian_2d` RS/OF tuning fits (Cholesky parameterisation).

Pick a cell with the **session dropdown** + **ROI slider** / **prev,next buttons** (or the **left/right arrow keys**). Sessions load lazily into an LRU cache (size configurable).

Per cell: measured RS/OF matrix, fitted matrix, residual map; depth / RS / OF tuning (data in black, fit in red); observed-vs-predicted; and the dataset test-R² distribution with the empirical-null threshold and this cell marked.

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')
from collections import OrderedDict
import numpy as np
import pandas as pd
import scipy.stats
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Javascript

import cottage_analysis.analysis.fit_gaussian_blob as fit_gb
from cottage_analysis.plotting import rsof_plots
from cottage_analysis.pipelines import pipeline_utils

NEURONS_DF_PATH = '/camp/home/blota/code/v1_depth_map/neurons_df_trial_average.pickle'
POPT_COL = 'rsof_popt_closedloop_g2d_treadmill_trial_average'
TEST_RSQ_COL = 'rsof_test_rsq_closedloop_g2d_treadmill_trial_average'
MIN_SIGMA = 0.25
PERCENTILE = 95
RS_THR = 0.01
MAX_RS2MOTOR_DIFF = 0.3
MAX_CACHE_DEFAULT = 10
LOG_RANGE = {'rs_bin_log_min': 0, 'rs_bin_log_max': 2.5, 'rs_bin_num': 6,
             'of_bin_log_min': -1.5, 'of_bin_log_max': 3.5, 'of_bin_num': 11,
             'log_base': 10}

In [ ]:
neurons_df = pd.read_pickle(NEURONS_DF_PATH)

def _is_popt(x):
    return isinstance(x, (list, np.ndarray)) and len(np.asarray(x).ravel()) >= 7 \
        and np.all(np.isfinite(np.asarray(x, dtype=float)))

ok = neurons_df[POPT_COL].apply(_is_popt)
neurons_df['g2d_angle'] = np.nan
neurons_df['g2d_ecc'] = np.nan
neurons_df.loc[ok, 'g2d_angle'] = neurons_df.loc[ok, POPT_COL].apply(
    lambda p: fit_gb.get_gaussian_angle(p, MIN_SIGMA))
neurons_df.loc[ok, 'g2d_ecc'] = neurons_df.loc[ok, POPT_COL].apply(
    lambda p: fit_gb.get_gaussian_eccentricity(p, MIN_SIGMA))

def empirical_null_threshold(rsq_values, percentile=95):
    vals = np.asarray(rsq_values, dtype=float)
    vals = vals[np.isfinite(vals)]
    neg = vals[vals < 0]
    sigma = np.std(neg)
    return scipy.stats.norm.ppf(percentile / 100, loc=0, scale=sigma), sigma

rsq_all = pd.to_numeric(neurons_df[TEST_RSQ_COL], errors='coerce').values
rsq_finite = rsq_all[np.isfinite(rsq_all) & (rsq_all >= -1)]
RSQ_THR, RSQ_SIGMA = empirical_null_threshold(rsq_finite, PERCENTILE)

sessions = sorted(neurons_df['session'].dropna().unique().tolist())
sess_to_project = neurons_df.groupby('session')['project'].first().to_dict()
print(f'{len(neurons_df)} cells, {len(sessions)} sessions, '
      f'R2 threshold={RSQ_THR:.3f} (sigma={RSQ_SIGMA:.3f})')

In [ ]:
_session_cache = OrderedDict()   # (project, session) -> trials_df
MAX_CACHE = {'n': MAX_CACHE_DEFAULT}

def _photodiode(session):
    return 2 if ('PZAH6.4b' in session or 'PZAG3.4f' in session) else 5

def _evict():
    while len(_session_cache) > max(1, MAX_CACHE['n']):
        _session_cache.popitem(last=False)

def load_session_trials(project, session):
    key = (project, session)
    if key in _session_cache:
        _session_cache.move_to_end(key)
        return _session_cache[key]
    _, _, _, trials_df = pipeline_utils.load_session(
        project=project, session_name=session,
        photodiode_protocol=_photodiode(session), regenerate_frames=False,
        filter_datasets=dict(annotated=True), protocol_base='SpheresTubeMotor')
    trials_df = trials_df[~trials_df.recording_name.str.contains('multidepth')]
    _session_cache[key] = trials_df
    _session_cache.move_to_end(key)
    _evict()
    return trials_df

def reconstruct_trials(trials_df, roi):
    """Per-trial averaged (rs_log, of_log_deg, dff, depth) for one ROI,
    replicating process_rs_of_for_fit (closed loop, running mask, trial_average)."""
    tdf = trials_df[trials_df.closed_loop == 1]
    rs_l, of_l, dff_l, depth_l = [], [], [], []
    has_diff = 'max_abs_rs2motor_diff_ratio_stim' in tdf.columns
    for _, tr in tdf.iterrows():
        rs = np.asarray(tr['RS_stim'], dtype=float)
        rse = np.asarray(tr['RS_eye_stim'], dtype=float)
        of = np.asarray(tr['OF_stim'], dtype=float)
        dff = np.asarray(tr['dff_stim'], dtype=float)[:, roi]
        run = (rs > RS_THR) & (rse > RS_THR) & (~np.isnan(of)) & (of > 0)
        if has_diff:
            run = run & (np.asarray(tr['max_abs_rs2motor_diff_ratio_stim'],
                                    dtype=float) < MAX_RS2MOTOR_DIFF)
        if run.sum() == 0:
            continue
        rs_l.append(np.mean(rs[run]))
        of_l.append(np.mean(of[run]))
        dff_l.append(np.mean(dff[run]))
        depth_l.append(tr['depth'])
    rs_l = np.array(rs_l); of_l = np.array(of_l)
    dff_l = np.array(dff_l); depth_l = np.array(depth_l)
    rs_log = np.log(rs_l)
    of_log = np.log(np.degrees(of_l))
    return rs_log, of_log, dff_l, depth_l

In [ ]:
def _tuning_panel(ax, x, dff, pred, bins, xlabel):
    if not len(dff):
        ax.set_xlabel(xlabel); ax.set_ylabel('dff'); return
    ax.scatter(x, dff, s=10, color='0.6', alpha=0.5, label='data')
    ax.scatter(x, pred, s=10, color='tab:red', alpha=0.4, marker='x', label='fit')
    om, edges, _ = scipy.stats.binned_statistic(x, dff, statistic='mean', bins=bins)
    pm, _, _ = scipy.stats.binned_statistic(x, pred, statistic='mean', bins=bins)
    ctr = np.sqrt(edges[:-1] * edges[1:])
    ax.plot(ctr, om, '-o', color='black', lw=1.5, ms=3, label='data mean')
    ax.plot(ctr, pm, '-o', color='tab:red', lw=1.5, ms=3, label='fit mean')
    ax.set_xscale('log'); ax.set_xlabel(xlabel); ax.set_ylabel('dff')
    ax.legend(fontsize=6)

def make_cell_figure(project, session, roi):
    row = neurons_df[(neurons_df.session == session) & (neurons_df.roi == roi)]
    fig = plt.figure(figsize=(15, 12))
    gs = fig.add_gridspec(3, 3, hspace=0.38, wspace=0.32)
    if len(row) == 0:
        fig.add_subplot(gs[0, 0]).text(0.5, 0.5, 'no matching row', ha='center')
        return fig
    row = row.iloc[0]
    popt = np.asarray(row[POPT_COL], dtype=float)
    session_df = neurons_df[neurons_df.session == session]
    trials_df = load_session_trials(project, session)

    ax_meas = fig.add_subplot(gs[0, 0]); ax_fit = fig.add_subplot(gs[0, 1])
    ax_res = fig.add_subplot(gs[0, 2])
    ax_depth = fig.add_subplot(gs[1, 0]); ax_rs = fig.add_subplot(gs[1, 1])
    ax_of = fig.add_subplot(gs[1, 2])
    ax_obs = fig.add_subplot(gs[2, 0]); ax_rsq = fig.add_subplot(gs[2, 1])
    ax_info = fig.add_subplot(gs[2, 2])

    vmax = None
    try:
        _, vmax, _ = rsof_plots.plot_RS_OF_matrix(
            trials_df, roi, log_range=LOG_RANGE, is_closed_loop=1,
            ax=ax_meas, title='Measured', return_matrix=True)
    except Exception as e:
        ax_meas.set_title(f'measured error: {e}')
    try:
        rsof_plots.plot_RS_OF_fit(
            session_df, roi, model='g2d', sfx='_treadmill_trial_average',
            min_sigma=MIN_SIGMA, vmin=0, vmax=vmax, log_range=LOG_RANGE,
            ax=ax_fit, label_r2=False)
        ax_fit.set_title('Fit')
    except Exception as e:
        ax_fit.set_title(f'fit error: {e}')

    rs_log, of_log, dff, depth = reconstruct_trials(trials_df, roi)
    pred = fit_gb.gaussian_2d((rs_log, of_log), *popt, min_sigma=MIN_SIGMA)
    resid = dff - pred
    rs_cm = np.exp(rs_log) * 100
    of_deg = np.exp(of_log)

    # residual map (same bins/extent as measured)
    rs_bins = np.insert(np.logspace(LOG_RANGE['rs_bin_log_min'],
                        LOG_RANGE['rs_bin_log_max'], LOG_RANGE['rs_bin_num'],
                        base=10), 0, 0)
    of_bins = np.insert(np.logspace(LOG_RANGE['of_bin_log_min'],
                        LOG_RANGE['of_bin_log_max'], LOG_RANGE['of_bin_num'],
                        base=10), 0, 0)
    extent = [LOG_RANGE['rs_bin_log_min'], LOG_RANGE['rs_bin_log_max'],
              LOG_RANGE['of_bin_log_min'], LOG_RANGE['of_bin_log_max']]
    if len(rs_cm):
        rmean, _, _, _ = scipy.stats.binned_statistic_2d(
            rs_cm, of_deg, resid, statistic='mean', bins=[rs_bins, of_bins])
        core = rmean[1:, 1:]
        vlim = np.nanmax(np.abs(core)) if np.isfinite(core).any() else 1.0
        im = ax_res.imshow(core.T, origin='lower', aspect='equal', cmap='RdBu_r',
                           vmin=-vlim, vmax=vlim, extent=extent)
        fig.colorbar(im, ax=ax_res, fraction=0.046, pad=0.04)
    ax_res.set_title('Residual (data - fit)')
    ax_res.set_xlabel('log10 RS'); ax_res.set_ylabel('log10 OF')

    # depth tuning (data vs fit)
    if len(dff):
        depth_cm = depth * 100
        uniq = np.unique(depth_cm)
        om = np.array([np.nanmean(dff[depth_cm == d]) for d in uniq])
        pm = np.array([np.nanmean(pred[depth_cm == d]) for d in uniq])
        ax_depth.scatter(depth_cm, dff, s=10, color='0.6', alpha=0.5, label='data')
        ax_depth.scatter(depth_cm, pred, s=10, color='tab:red', alpha=0.4,
                         marker='x', label='fit')
        ax_depth.plot(uniq, om, '-o', color='black', lw=1.5, ms=4, label='data mean')
        ax_depth.plot(uniq, pm, '-o', color='tab:red', lw=1.5, ms=4, label='fit mean')
        ax_depth.set_xscale('log'); ax_depth.legend(fontsize=6)
    ax_depth.set_xlabel('Virtual depth (cm)'); ax_depth.set_ylabel('dff')
    ax_depth.set_title('Depth tuning')

    # RS and OF tuning (data vs fit)
    _tuning_panel(ax_rs, rs_cm, dff, pred, np.logspace(0, 2.5, 12),
                  'Running speed (cm/s)')
    ax_rs.set_title('RS tuning')
    _tuning_panel(ax_of, of_deg, dff, pred, np.logspace(-1.5, 3.5, 12),
                  'Optic flow (deg/s)')
    ax_of.set_title('OF tuning')

    # observed vs predicted
    r2_all = np.nan
    if len(dff):
        ax_obs.scatter(pred, dff, s=12, alpha=0.5)
        lo = float(np.nanmin([pred.min(), dff.min()]))
        hi = float(np.nanmax([pred.max(), dff.max()]))
        ax_obs.plot([lo, hi], [lo, hi], 'k--', lw=1)
        ss_res = np.nansum((dff - pred) ** 2)
        ss_tot = np.nansum((dff - np.nanmean(dff)) ** 2)
        r2_all = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
    test_rsq = pd.to_numeric(pd.Series([row[TEST_RSQ_COL]]), errors='coerce').iloc[0]
    ax_obs.set_xlabel('predicted'); ax_obs.set_ylabel('observed')
    ax_obs.set_title(f'R2(all)={r2_all:.3f}  test R2={test_rsq:.3f}')

    # dataset test-R2 distribution with threshold + this cell
    ax_rsq.hist(rsq_finite, bins=80, color='lightgray')
    ax_rsq.axvline(RSQ_THR, color='red', ls='--', label=f'thr={RSQ_THR:.3f}')
    if np.isfinite(test_rsq):
        ax_rsq.axvline(test_rsq, color='green', lw=2, label=f'cell={test_rsq:.3f}')
    ax_rsq.set_xlabel('test R2'); ax_rsq.set_ylabel('count')
    ax_rsq.set_title('Dataset test R2'); ax_rsq.legend(fontsize=6)

    # info
    ax_info.axis('off')
    pref_rs = row.get('preferred_RS_closedloop_g2d_treadmill_trial_average', np.nan)
    info = (f'session : {session}\n'
            f'roi     : {roi}\n'
            f'project : {project}\n'
            f'n trials: {len(dff)}\n'
            f'ecc     : {row["g2d_ecc"]:.3f}\n'
            f'angle   : {row["g2d_angle"]:.1f} deg\n'
            f'pref RS : {pref_rs * 100:.1f} cm/s\n'
            f'test R2 : {test_rsq:.3f}\n'
            f'thr     : {RSQ_THR:.3f}\n'
            f'sig     : {test_rsq > RSQ_THR}\n'
            f'depth-tuned: {row.get("is_depth_neuron", np.nan)}')
    ax_info.text(0.02, 0.98, info, va='top', ha='left', fontsize=11,
                 family='monospace')

    fig.suptitle(f'{session}   roi {roi}', fontsize=13)
    return fig

In [ ]:
session_dd = widgets.Dropdown(options=sessions, description='session',
                              layout=widgets.Layout(width='320px'))
prev_btn = widgets.Button(description='◀ prev', layout=widgets.Layout(width='90px'))
roi_sl = widgets.IntSlider(min=0, max=1, description='roi',
                           continuous_update=False,
                           layout=widgets.Layout(width='300px'))
next_btn = widgets.Button(description='next ▶', layout=widgets.Layout(width='90px'))
prev_btn.add_class('roi-prev')
next_btn.add_class('roi-next')
cache_it = widgets.BoundedIntText(value=MAX_CACHE_DEFAULT, min=1, max=50,
                                  description='max cache',
                                  layout=widgets.Layout(width='160px'))
status = widgets.HTML()
out = widgets.Output()
state = {'project': None, 'session': None, 'roi': None, 'busy': False}

def session_rois(session):
    return sorted(neurons_df[neurons_df.session == session]['roi']
                  .dropna().astype(int).unique().tolist())

def render_cell():
    with out:
        out.clear_output(wait=True)
        fig = make_cell_figure(state['project'], state['session'], state['roi'])
        display(fig)
        plt.close(fig)

def select(project, session, roi):
    if state['busy']:
        return
    state['busy'] = True
    try:
        state.update(project=project, session=session, roi=int(roi))
        if session_dd.value != session:
            session_dd.value = session
        rois = session_rois(session)
        roi_sl.min = min(rois); roi_sl.max = max(rois)
        if roi_sl.value != roi:
            roi_sl.value = int(roi)
        if (project, session) not in _session_cache:
            status.value = f'<b style="color:darkorange">loading {session} …</b>'
        render_cell()
        status.value = f'cached sessions: {len(_session_cache)} / {MAX_CACHE["n"]}'
    finally:
        state['busy'] = False

def step_roi(delta):
    if state['busy']:
        return
    rois = session_rois(state['session'])
    if not rois:
        return
    i = rois.index(state['roi']) if state['roi'] in rois else 0
    j = min(max(i + delta, 0), len(rois) - 1)
    select(state['project'], state['session'], rois[j])

def on_session(ch):
    if state['busy']:
        return
    s = ch['new']
    select(sess_to_project[s], s, session_rois(s)[0])

def on_roi(ch):
    if state['busy']:
        return
    select(state['project'], state['session'], ch['new'])

def on_cache(ch):
    MAX_CACHE['n'] = ch['new']
    _evict()
    status.value = f'cached sessions: {len(_session_cache)} / {MAX_CACHE["n"]}'

prev_btn.on_click(lambda b: step_roi(-1))
next_btn.on_click(lambda b: step_roi(+1))
session_dd.observe(on_session, 'value')
roi_sl.observe(on_roi, 'value')
cache_it.observe(on_cache, 'value')

controls = widgets.HBox([session_dd, prev_btn, roi_sl, next_btn, cache_it])
dashboard = widgets.VBox([controls, status, out])
display(dashboard)

# Left/Right arrow keys -> previous/next ROI.
display(Javascript('''
document.addEventListener('keydown', function(e){
  if(e.target && (e.target.tagName==='INPUT' || e.target.tagName==='TEXTAREA')) return;
  if(e.key==='ArrowLeft'){ var b=document.querySelector('.roi-prev'); if(b){b.click(); e.preventDefault();} }
  else if(e.key==='ArrowRight'){ var b=document.querySelector('.roi-next'); if(b){b.click(); e.preventDefault();} }
});
'''))

_s0 = sessions[0]
select(sess_to_project[_s0], _s0, session_rois(_s0)[0])

## Running this viewer with Voila

On the compute node (where `/nemo` is native — no Samba):

```bash
conda activate v1_depth_map
cd /camp/home/blota/home/users/blota/code/v1_depth_map/v1_depth_map/presentations
voila fit_quality_viewer.ipynb --no-browser --port=8866 --Voila.ip=127.0.0.1
```

From your laptop (`camp_int` points at the active node):

```bash
ssh -L 8866:localhost:8866 camp_int
```

then open <http://localhost:8866>.

**Notes**
- First selection in a session triggers a (slow) `load_session`; later ROIs in that session are instant. Up to *max cache* sessions stay in memory (LRU).
- Navigate ROIs with the slider, the prev/next buttons, or the left/right arrow keys.